## Initial Feature Selection

We conduct initial feature selection, with our main consideration being to ensure we don't include any correlated variables, and if we can hypothesize any possible explanations as to how a variable is related to churn risk.

#### Customers

For instance, in the Customers table, the variables partner and dependents could be argued to be correlated to the age variable. Older people are more likely to have a partner and dependents, and more likely to prefer certain payment methods over others. Hence, you could represent both of those features using just one variable: age.

So, from this table we will use the variables: **gender, age, region**

For age, maybe being older makes you less likely to switch. One reason perhaps is having dependents means switching providers means also switching your dependents' providers as well.

For gender, you can extend the same logic for age, as men are more likely to be breadwinners and have dependents.

For region, we want to investigate if service in a given region can influence churn risk.

#### Data Service 

Here, a key feature is data_use. Higher data usage correlates to a higher monthly charge, as well as to what types of services they may have purchased. A higher data usage indicates that they have purchased an internet plan, while lower usage indicates they have only purchased a mobile plan. Hence, data_use can be used to encompass multiple variables.

We can also include total months as a variable in our model, but that means that we cannot simultaneously have monthly_bill and total_bill as that would cause perfect multicollinearity. (Due to total_bill = monthly_bill x total months.)

So from this table we'll use the **total months** and **data_use** variables in our model.

Since some ids show up more than once, we'll use aggregation to give us maximum total months i.e. the most recent value of total months and as well as the most recent value of data_use.

The thinking goes that the longer a customer's total months, the less likely they will churn as they may be seen as "loyal" to the company.

And a customer with higher data usage, could potentially be seen as loyal as well. This would be especially true if certain services like internet require infrastructure like modems to be used, which makes switching to different providers more costly.

## Data Cleaning

#### Customer Base

Just simple value standardization and data type conversion.

In [15]:
import pandas as pd
import numpy as np

#Customer Base
customers = pd.read_csv('customers.csv')

#Turn churn into a binary value, where 1 means the customer has churned
customers["churn"] = [1 if type(x) == str else 0 for x in customers['churn_date']]

#Region has the following values: {'CHI', 'Chicago', 'Los Angeles', 'Miami', 'NYC', 'New York City', 'Pennsylvania', 'lax', 'los angeles', 'new york city'}
#Now we will standardize the values

customers["region"] = customers["region"].replace({
                                                    "NYC": "New York City",
                                                    "new york city": "New York City",
                                                    "CHI":"Chicago",
                                                    'los angeles': "Los Angeles",
                                                    'lax': "Los Angeles",
                                                })


#Remove NaNs and convert dates from str to timestamp
customers['churn_date'] = pd.to_datetime(customers['churn_date'], errors='coerce', dayfirst=True).dt.date
customers['churn_date'] = customers["churn_date"].fillna("Not yet churned")


#### Service Usage

Here we use the most recent total months and most recent data_use.

Average data usage was considered but since most customers only had 1-2 entries, average probably is not necessary.

In [16]:
#Service Usage
data_service = pd.read_csv('data service.csv')

#Get max total months, which also gives most recent row
data_service_total_months = data_service.groupby("id")["total_months"].max().reset_index()

#Use only data from most recent
data_service = data_service.merge(data_service_total_months, on=["id","total_months"])

#Merge with data_service
customers = customers.merge(data_service[["id","total_months","data_use"]])

## Training

#### Final Data Cleaning

Final data cleaning before training, by dropping NAs. Many of the values that are NA are categorical which makes imputing more
difficult. Also not that many NA rows so even after removal still plenty of rows left for 
training.

In [17]:
customers.dropna(inplace=True)

#### Train Test Split

In [18]:
from sklearn.model_selection import train_test_split

X = customers[["gender", "age", "region", "total_months", "data_use"]]

#Get dummy values for categorical variables                   
X = pd.get_dummies(X, drop_first=True)

y = customers["churn"]
 
#Use a 70-30 split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=52)

#### Logistic Regression

As this is a classification problem, logistic regression is chosen as the model. It is also chosen because logistic regression provides interpretable coefficients, which can be used to glean insights and shape business strategy.

In [19]:
import statsmodels.api as sm

logreg = sm.Logit(y_train, X_train.astype(float)).fit()

         Current function value: 0.148326
         Iterations: 35


c:\Rafid\Work\Customer Churn Analysis\data\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


A summary of the model is provided below.

In [20]:
logreg.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                  churn   No. Observations:                  473
Model:                          Logit   Df Residuals:                      465
Method:                           MLE   Df Model:                            7
Date:                Fri, 27 Mar 2026   Pseudo R-squ.:                  0.7226
Time:                        14:02:22   Log-Likelihood:                -70.158
converged:                      False   LL-Null:                       -252.90
Covariance Type:            nonrobust   LLR p-value:                 5.984e-75
========================================================================================
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
age                      0.1037      0.026      4.047      0.000       0.053       0.154
total_months            -0.7150      0.123     -5.824      0.000      -0.956      -0.474
data_use                 0.0082      0.004      2.075      0.038       0.000       0.016
gender_Male              0.3033      0.429      0.707      0.479      -0.537       1.144
region_Los Angeles      -0.9088      0.560     -1.622      0.105      -2.007       0.189
region_Miami           -35.9020   9.69e+06   -3.7e-06      1.000    -1.9e+07     1.9e+07
region_New York City    -1.4108      0.659     -2.139      0.032      -2.703      -0.118
region_Pennsylvania     -2.7491      0.772     -3.561      0.000      -4.262      -1.236
========================================================================================

Possibly complete quasi-separation: A fraction 0.57 of observations can be
perfectly predicted. This might indicate that there is complete
quasi-separation. In this case some parameters will not be identified.
"""

In [21]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

y_pred = logreg.predict(X_train.astype(float)).round()

print("Precision score:", precision_score(y_pred, y_train))
print("Recall score:", recall_score(y_pred, y_train))

Precision score: 0.9065420560747663
Recall score: 0.8508771929824561


In [22]:
confusion_matrix(y_pred, y_train)

array([[349,  10],
       [ 17,  97]])

The model performs quite well. The precision and recall scores are both quite high in the 90s and mid 80s. This is important because the dataset has a class imbalance, with churned customers representing only around 20% of all entries. With a class imbalance, attaining a high accuracy is much easier as it can be achieved by predicting the majority class for all entries.

In order for a model to be useful, it needs to be discriminatory, meaning it should have good precision and recall scores.

Precision measures how many of the positive detections were true. Generally means there are few false positives.

Recall measures how many of the true positives were detected appropriately. Generally means there are few false negatives.

#### Second Round of Training

Despite performing well, we see that many variables are insignificant or have high errors. To prevent overfitting and improve interpretability we will be dropping some variables.

When dropping variables, we are using the significance values as well as model performance as guides to determine what is appropriate to drop or not. We also need to ensure we continue only using the train set to make decisions on which variables to drop.

The following variables were the ones we decided to drop, and the summary of the new model can be seen below.

In [23]:
#Dropped variables.
dropped_vars = ["region_Miami", "gender_Male"]

#Get new X values based on dropped vars
new_X_train = X_train.drop(dropped_vars, axis=1)
new_X_test = X_test.drop(dropped_vars, axis=1)

logreg2 = sm.Logit(y_train, new_X_train.astype(float)).fit()

logreg2.summary()

Optimization terminated successfully.
         Current function value: 0.161370
         Iterations 12


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                  churn   No. Observations:                  473
Model:                          Logit   Df Residuals:                      467
Method:                           MLE   Df Model:                            5
Date:                Fri, 27 Mar 2026   Pseudo R-squ.:                  0.6982
Time:                        14:02:22   Log-Likelihood:                -76.328
converged:                       True   LL-Null:                       -252.90
Covariance Type:            nonrobust   LLR p-value:                 3.698e-74
========================================================================================
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
age                      0.0780      0.022      3.536      0.000       0.035       0.121
total_months            -0.6416      0.112     -5.737      0.000      -0.861      -0.422
data_use                 0.0104      0.005      2.172      0.030       0.001       0.020
region_Los Angeles      -0.5396      0.510     -1.058      0.290      -1.539       0.460
region_New York City    -0.8527      0.586     -1.456      0.145      -2.001       0.295
region_Pennsylvania     -2.0269      0.700     -2.898      0.004      -3.398      -0.656
========================================================================================

Possibly complete quasi-separation: A fraction 0.55 of observations can be
perfectly predicted. This might indicate that there is complete
quasi-separation. In this case some parameters will not be identified.
"""

In [24]:
y_pred = logreg2.predict(new_X_train.astype(float)).round()

print("Precision score:", precision_score(y_pred, y_train))
print("Recall score:", recall_score(y_pred, y_train))

Precision score: 0.8878504672897196
Recall score: 0.8482142857142857


In [25]:
confusion_matrix(y_pred, y_train)

array([[349,  12],
       [ 17,  95]])

As we can see, the dropped variables led to a decrease in the precision, but not a severe one and it was achieved with fewer variables and the remaining variables have far less severe variance than in the first model.

#### Test Set Performance

The model also performs well on the test set. Precision and recall remain in the 90s and 80s respectively and there are few false positives and negatives, showing good discrimination.

In [26]:
y_pred = logreg2.predict(new_X_test.astype(float)).round()

print("Precision score:", precision_score(y_pred, y_test))
print("Recall score:", recall_score(y_pred, y_test))

Precision score: 0.9210526315789473
Recall score: 0.8333333333333334


In [27]:
confusion_matrix(y_pred, y_test)

array([[159,   3],
       [  7,  35]])

## Model Insights

In [28]:
logreg2.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                  churn   No. Observations:                  473
Model:                          Logit   Df Residuals:                      467
Method:                           MLE   Df Model:                            5
Date:                Fri, 27 Mar 2026   Pseudo R-squ.:                  0.6982
Time:                        14:02:22   Log-Likelihood:                -76.328
converged:                       True   LL-Null:                       -252.90
Covariance Type:            nonrobust   LLR p-value:                 3.698e-74
========================================================================================
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
age                      0.0780      0.022      3.536      0.000       0.035       0.121
total_months            -0.6416      0.112     -5.737      0.000      -0.861      -0.422
data_use                 0.0104      0.005      2.172      0.030       0.001       0.020
region_Los Angeles      -0.5396      0.510     -1.058      0.290      -1.539       0.460
region_New York City    -0.8527      0.586     -1.456      0.145      -2.001       0.295
region_Pennsylvania     -2.0269      0.700     -2.898      0.004      -3.398      -0.656
========================================================================================

Possibly complete quasi-separation: A fraction 0.55 of observations can be
perfectly predicted. This might indicate that there is complete
quasi-separation. In this case some parameters will not be identified.
"""

#### Variable Interpretation

**age**: The older the customer the more likely they are to churn.

**total_months**: The negative coefficient indicates that the longer a customer's total months, the less likely they are to churn.  

**data_use**: The positive coefficient shows that the higher the data usage, the more likely to churn.

**region**: Notably, customers in Los Angeles, New York and Pennsylvania are less likely to churn compared to baseline, which are the customers in Miami and Chicago.

##### Retention Strategy

The older a customer, the likelihood of churning goes up for them, so the company should be monitor usage from older users and be more wary with them.

An interesting result was that of data usage. If higher data usage indicates that there is a higher risk of churn, it might be a possible indicator that this company's high data offerings are somewhat lackluster in comparison to other providers, so that is something worth investigating in upcoming product improvement inititative.

Similar investigations should also be done about service in Miami and Chicago, see why customers in Pennsylvania, Los Angeles and New York City are less likely to churn compared to customers in the former cities.

In fact, generally speaking, this company needs to ensure that it provides a seamless, problem free experience for all customers because the worry is the moment they experience technical issues, their risk of churn might go up. 

The company could also consider customer incentives to have long total months and reward customer loyalty such as through loyalty programs to further disincentivize customers from switching providers.

Those are some general insights, but we can also use the predictive model to be even more proactive at preventing churn. When using the model and we detect a customer is at risk of churn, we can offer them retention incentives.

The value of the incentive should be dependent on the *potential* value we can gain from retaining said customer. A customer with high value should be one with long total months and high data usage. Such a customer should receive a premium incentive package. Meanwhile, a customer with lower total months and lower data usage would receive a more basic package, or maybe not even one at all if their potential value isn't worth it.

## Limitations, and Recommendations for Further Research
In the exploratory phase, correlation was mainly determined through domain knowledge. Should have backed it up with more statistically robust methods as well such as VIF. Not done here because many of the values were categorical, with multiple classes, which makes VIF harder to apply and beyond the scope of expertise.

Perhaps could also explore more statistically robust methods of feature elimination.

In this analysis, interaction variables were not made use of. In future analysis, such interactions should be considered.

Should conduct further research to determine potential value of a customer, to determine the appropriate value for the incentive packages.

Also try and investigate if more complex models offer any marked improvement compared to this simple logistic regression model.